In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('ticks', rc={'axes.facecolor': (0, 0, 0, 0)})
sns.set_context('talk')

from matplotlib import rcParams, colors, cm
rcParams['font.family'] = 'sans-serif'
rcParams['figure.dpi'] = 150

pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 100)

In [47]:
df = pd.read_excel("../../data/Metadata.xlsx")

In [48]:
df

,Short_name,Condition,Replicate,Measured_OD,Desired_OD,Medium,used_C-source,Project,Organism,Strain,Reference_Srain,Sample_type,Quantification,Conc.[ng/μl],Volume[μl],Ratio_260/280,Ratio_260/230
0,D39_lac_1,Lactose_120,1,0.123,0.12,Lacks,Lactose [25mM],bacterial RNASeq,S. pnreumoniae,D39,GCF_000014365.2,total RNA,Nanodrop,79.499,25,2.019,1.270
1,D39_lac_2,Lactose_120,2,0.121,0.12,Lacks,Lactose [25mM],bacterial RNASeq,S. pnreumoniae,D39,GCF_000014365.2,total RNA,Nanodrop,86.279,25,1.923,0.995
2,D39_lac_3,Lactose_120,3,0.120,0.12,Lacks,Lactose [25mM],bacterial RNASeq,S. pnreumoniae,D39,GCF_000014365.2,total RNA,Nanodrop,87.707,25,2.025,1.328
3,D39_lac_4,Lactose_400,1,0.411,0.40,Lacks,Lactose [25mM],bacterial RNASeq,S. pnreumoniae,D39,GCF_000014365.2,total RNA,Nanodrop,73.177,25,1.956,1.039
4,D39_lac_5,Lactose_400,2,0.414,0.40,Lacks,Lactose [25mM],bacterial RNASeq,S. pnreumoniae,D39,GCF_000014365.2,total RNA,Nanodrop,74.331,25,1.942,0.587
5,D39_lac_6,Lactose_400,3,0.416,0.40,Lacks,Lactose [25mM],bacterial RNASeq,S. pnreumoniae,D39,GCF_000014365.2,total RNA,Nanodrop,145.104,25,1.863,0.606
6,D39_LNnT_7,LNnT_120,1,0.124,0.12,Lacks,LNnT [25mM],bacterial RNASeq,S. pnreumoniae,D39,GCF_000014365.2,total RNA,Nanodrop,16.215,25,1.946,0.360
7,D39_LNnT_8,LNnT_120,2,0.119,0.12,Lacks,LNnT [25mM],bacterial RNASeq,S. pnreumoniae,D39,GCF_000014365.2,total RNA,Nanodrop,12.272,25,2.160,0.021
8,D39_LNnT_9,LNnT_120,3,0.115,0.12,Lacks,LNnT [25mM],bacterial RNASeq,S. pnreumoniae,D39,GCF_000014365.2,total RNA,Nanodrop,12.258,25,1.779,0.217
9,D39_LNnT_10,LNnT_400,1,0.412,0.40,Lacks,LNnT [25mM],bacterial RNASeq,S. pnreumoniae,D39,GCF_000014365.2,total RNA,Nanodrop,42.765,25,1.812,0.912


### Prepare conditions

In [49]:
conds = df['Condition'].drop_duplicates().to_numpy()

i, j = np.triu_indices(len(conds), k=1)

conditions = pd.DataFrame({
    "Condition1": conds[i],
    "Condition2": conds[j]
})

In [50]:
conditions

,Condition1,Condition2
0,Lactose_120,Lactose_400
1,Lactose_120,LNnT_120
2,Lactose_120,LNnT_400
3,Lactose_120,6SL_120
4,Lactose_120,6SL_400
5,Lactose_400,LNnT_120
6,Lactose_400,LNnT_400
7,Lactose_400,6SL_120
8,Lactose_400,6SL_400
9,LNnT_120,LNnT_400


remove conditions, 10-13

In [51]:
conditions = conditions.drop(conditions.index[10:14])

In [52]:
conditions

,Condition1,Condition2
0,Lactose_120,Lactose_400
1,Lactose_120,LNnT_120
2,Lactose_120,LNnT_400
3,Lactose_120,6SL_120
4,Lactose_120,6SL_400
5,Lactose_400,LNnT_120
6,Lactose_400,LNnT_400
7,Lactose_400,6SL_120
8,Lactose_400,6SL_400
9,LNnT_120,LNnT_400


In [53]:
# reorder and reset index
order = [0, 14, 9, 1, 3, 6, 8, 2, 4, 5, 7]
conditions = conditions.loc[order].reset_index(drop=True)

In [54]:
conditions

,Condition1,Condition2
0,Lactose_120,Lactose_400
1,6SL_120,6SL_400
2,LNnT_120,LNnT_400
3,Lactose_120,LNnT_120
4,Lactose_120,6SL_120
5,Lactose_400,LNnT_400
6,Lactose_400,6SL_400
7,Lactose_120,LNnT_400
8,Lactose_120,6SL_400
9,Lactose_400,LNnT_120


In [62]:
conditions.to_csv("../../data/conditions.tsv", index=None, sep="\t")

### Prepare samples

In [56]:
samples = df[["Short_name", "Condition", "Replicate"]]

In [58]:
reads_path = "../../data/reads"

samples["file1"] = samples["Short_name"] + "_R1.fastq.gz"
samples["file2"] = samples["Short_name"] + "_R2.fastq.gz"

/scratch/local/47295906/ipykernel_1074583/2637239392.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  samples["file1"] = samples["Short_name"] + "_R1.fastq.gz"
/scratch/local/47295906/ipykernel_1074583/2637239392.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  samples["file2"] = samples["Short_name"] + "_R2.fastq.gz"


In [59]:
samples.rename(columns={"Short_name":"sample", "Condition":"group", "Replicate":"rep_no"}, inplace=True)
samples = samples[["sample", "file1", "file2", "group", "rep_no"]]

/scratch/local/47295906/ipykernel_1074583/2674797129.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  samples.rename(columns={"Short_name":"sample", "Condition":"group", "Replicate":"rep_no"}, inplace=True)


In [60]:
samples

,sample,file1,file2,group,rep_no
0,D39_lac_1,D39_lac_1_R1.fastq.gz,D39_lac_1_R2.fastq.gz,Lactose_120,1
1,D39_lac_2,D39_lac_2_R1.fastq.gz,D39_lac_2_R2.fastq.gz,Lactose_120,2
2,D39_lac_3,D39_lac_3_R1.fastq.gz,D39_lac_3_R2.fastq.gz,Lactose_120,3
3,D39_lac_4,D39_lac_4_R1.fastq.gz,D39_lac_4_R2.fastq.gz,Lactose_400,1
4,D39_lac_5,D39_lac_5_R1.fastq.gz,D39_lac_5_R2.fastq.gz,Lactose_400,2
5,D39_lac_6,D39_lac_6_R1.fastq.gz,D39_lac_6_R2.fastq.gz,Lactose_400,3
6,D39_LNnT_7,D39_LNnT_7_R1.fastq.gz,D39_LNnT_7_R2.fastq.gz,LNnT_120,1
7,D39_LNnT_8,D39_LNnT_8_R1.fastq.gz,D39_LNnT_8_R2.fastq.gz,LNnT_120,2
8,D39_LNnT_9,D39_LNnT_9_R1.fastq.gz,D39_LNnT_9_R2.fastq.gz,LNnT_120,3
9,D39_LNnT_10,D39_LNnT_10_R1.fastq.gz,D39_LNnT_10_R2.fastq.gz,LNnT_400,1


In [61]:
samples.to_csv("../../data/samples.tsv", index=None, sep="\t")